In [1]:
# # Composite DNA Decoder: Training & Evaluation - Extended Version
# ## Supports Multiple Error Models and Alphabets

# In[1]:

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID

# In[2]:

# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080


In [2]:
# In[3]:

# =============================================================================
# CELL 3: CONFIGURATION & HYPERPARAMETERS
# =============================================================================

# ------------------- SELECT ERROR MODEL -------------------
# Options: "erlich", "grass", "organick"
ERROR_MODEL = "grass"  # <-- CHANGE THIS

# ------------------- SELECT ALPHABET MODE -------------------
# Options: "2mix_only", "2mix_3mix", "2mix_3mix_4mix"
ALPHABET_MODE = "2mix_only"  # <-- CHANGE THIS
# ------------------------------------------------------------

# Dataset parameters
NUM_SAMPLES = 100000
MAX_COVERAGE = 25

# Error model specifications
ERROR_MODEL_SPECS = {
    "erlich": {
        "seq_length": 136,
        "name": "EZ17"
    },
    "grass": {
        "seq_length": 104,
        "name": "G15"
    },
    "organick": {
        "seq_length": 77,
        "name": "O17"
    }
}

# Vocabulary sizes
VOCAB_SIZES = {
    "2mix_only": 10,
    "2mix_3mix": 14,
    "2mix_3mix_4mix": 15
}

# Build configuration
CONFIG = {
    # Error Model
    "error_model": ERROR_MODEL,
    "error_name": ERROR_MODEL_SPECS[ERROR_MODEL]["name"],
    
    # Alphabet Mode
    "alphabet_mode": ALPHABET_MODE,
    
    # Data Paths
    "dataset_dir": "./dataset",
    "dataset_name": f"dna_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    
    # Results directory based on configuration
    "results_dir": f"./results_{ERROR_MODEL_SPECS[ERROR_MODEL]['name']}_{ALPHABET_MODE}",
    
    # Vocabulary
    "vocab_size": VOCAB_SIZES[ALPHABET_MODE],
    
    # Sequence Parameters
    "seq_length": ERROR_MODEL_SPECS[ERROR_MODEL]["seq_length"],
    
    # Experiment Parameters
    "coverage_levels": [1, 2, 3, 5, 8, 10, 15, 20, 25],
    
    # Model Architecture
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
    
    # Training Parameters
    "batch_size": 500,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "patience": 10,
    "warmup_epochs": 10,
    "min_lr": 1e-6,
    
    # Reproducibility
    "seed": 42
}

# Complete dataset path
CONFIG["dataset_path"] = (f"{CONFIG['dataset_dir']}/"
                          f"{CONFIG['dataset_name']}_"
                          f"{NUM_SAMPLES}_{MAX_COVERAGE}.pkl")

# Create results directory
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print(f"{'='*60}")
print(f"📋 CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Alphabet Mode: {CONFIG['alphabet_mode']}")
print(f"   Vocab Size: {CONFIG['vocab_size']} classes")
print(f"   Dataset Path: {CONFIG['dataset_path']}")
print(f"   Results Dir: {CONFIG['results_dir']}")
print(f"{'='*60}")

📋 CONFIGURATION
   Error Model: grass (G15)
   Sequence Length: 104
   Alphabet Mode: 2mix_only
   Vocab Size: 10 classes
   Dataset Path: ./dataset/dna_G15_2mix_only_100000_25.pkl
   Results Dir: ./results_G15_2mix_only


In [3]:
# =============================================================================
# CELL 4: SEED & REPRODUCIBILITY
# =============================================================================
def set_seed(seed):
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
print(f"🎲 Random seed set to: {CONFIG['seed']}")

🎲 Random seed set to: 42


In [4]:

# =============================================================================
# CELL 5: SYMBOL MAPPINGS & IDEAL VECTORS
# =============================================================================

def build_symbol_to_idx(mode):
    """Build symbol-to-index mapping based on alphabet mode."""
    # Pure bases: indices 0-3
    symbol_to_idx = {
        'A': 0, 'C': 1, 'G': 2, 'T': 3,
    }
    
    # Two-mix: indices 4-9
    symbol_to_idx.update({
        'M1': 4, 'M2': 5, 'M3': 6, 'M4': 7, 'M5': 8, 'M6': 9
    })
    
    # Three-mix: indices 10-13
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        symbol_to_idx.update({
            'T1': 10, 'T2': 11, 'T3': 12, 'T4': 13
        })
    
    # Four-mix: index 14
    if mode == "2mix_3mix_4mix":
        symbol_to_idx.update({
            'Q1': 14
        })
    
    return symbol_to_idx


def build_ideal_vectors(mode):
    """Build ideal frequency vectors for all symbols."""
    # [A_prob, C_prob, G_prob, T_prob]
    
    ideal_vectors = [
        # Pure bases (indices 0-3)
        [1.0, 0.0, 0.0, 0.0],  # A
        [0.0, 1.0, 0.0, 0.0],  # C
        [0.0, 0.0, 1.0, 0.0],  # G
        [0.0, 0.0, 0.0, 1.0],  # T
        
        # Two-mix (indices 4-9) - uniform 0.5/0.5
        [0.5, 0.0, 0.0, 0.5],  # M1 (A|T)
        [0.0, 0.5, 0.5, 0.0],  # M2 (C|G)
        [0.0, 0.5, 0.0, 0.5],  # M3 (C|T)
        [0.0, 0.0, 0.5, 0.5],  # M4 (G|T)
        [0.5, 0.5, 0.0, 0.0],  # M5 (A|C)
        [0.5, 0.0, 0.5, 0.0],  # M6 (A|G)
    ]
    
    # Three-mix (indices 10-13) - uniform 1/3 each
    if mode in ["2mix_3mix", "2mix_3mix_4mix"]:
        third = 1.0 / 3.0
        ideal_vectors.extend([
            [third, third, third, 0.0],    # T1 (A|C|G)
            [third, third, 0.0, third],    # T2 (A|C|T)
            [third, 0.0, third, third],    # T3 (A|G|T)
            [0.0, third, third, third],    # T4 (C|G|T)
        ])
    
    # Four-mix (index 14) - uniform 0.25 each
    if mode == "2mix_3mix_4mix":
        ideal_vectors.append(
            [0.25, 0.25, 0.25, 0.25]       # Q1 (A|C|G|T)
        )
    
    return torch.tensor(ideal_vectors, dtype=torch.float32)


# Build mappings
SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"]).to(device)

print(f"\n📊 Symbol Mappings ({CONFIG['alphabet_mode']}):")
print(f"   {'Symbol':<8} {'Index':<6} {'Ideal Vector [A, C, G, T]'}")
print(f"   {'-'*50}")
for sym, idx in sorted(SYMBOL_TO_IDX.items(), key=lambda x: x[1]):
    vec = IDEAL_VECTORS[idx].cpu().numpy()
    print(f"   {sym:<8} {idx:<6} [{vec[0]:.4f}, {vec[1]:.4f}, {vec[2]:.4f}, {vec[3]:.4f}]")



📊 Symbol Mappings (2mix_only):
   Symbol   Index  Ideal Vector [A, C, G, T]
   --------------------------------------------------
   A        0      [1.0000, 0.0000, 0.0000, 0.0000]
   C        1      [0.0000, 1.0000, 0.0000, 0.0000]
   G        2      [0.0000, 0.0000, 1.0000, 0.0000]
   T        3      [0.0000, 0.0000, 0.0000, 1.0000]
   M1       4      [0.5000, 0.0000, 0.0000, 0.5000]
   M2       5      [0.0000, 0.5000, 0.5000, 0.0000]
   M3       6      [0.0000, 0.5000, 0.0000, 0.5000]
   M4       7      [0.0000, 0.0000, 0.5000, 0.5000]
   M5       8      [0.5000, 0.5000, 0.0000, 0.0000]
   M6       9      [0.5000, 0.0000, 0.5000, 0.0000]


In [5]:
# =============================================================================
# CELL 6: DATA PREPROCESSING
# =============================================================================

def preprocess_cluster_to_matrix(cluster_reads, target_length):
    """
    Convert variable-length noisy reads into a (4, target_length) normalized frequency matrix.
    
    Args:
        cluster_reads: List of noisy DNA strings
        target_length: Target sequence length
    
    Returns:
        (4, target_length) numpy array with normalized nucleotide frequencies
    """
    profile_matrix = np.zeros((4, target_length), dtype=np.float32)
    base_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    num_reads = len(cluster_reads)
    
    for read in cluster_reads:
        read_len = len(read)
        if read_len == 0:
            continue
            
        # Alignment via linear interpolation
        for t_idx in range(target_length):
            read_idx = int((t_idx + 0.5) * (read_len / target_length))
            if read_idx >= read_len:
                read_idx = read_len - 1
            
            base = read[read_idx]
            if base in base_map:
                row_idx = base_map[base]
                profile_matrix[row_idx, t_idx] += 1.0
                
    if num_reads > 0:
        profile_matrix /= num_reads
        
    return profile_matrix


In [6]:
# =============================================================================
# CELL 7: PYTORCH DATASET CLASS
# =============================================================================

class CompositeDNADataset(Dataset):
    """
    PyTorch Dataset for Composite DNA data.
    Adapts to different sequence lengths based on error model.
    """
    def __init__(self, data_path, seq_length, symbol_to_idx, limit_coverage=None):
        with open(data_path, 'rb') as f:
            raw_data = pickle.load(f)
        self.samples = raw_data['data']
        self.metadata = raw_data['metadata']
        self.seq_length = seq_length  # Variable based on error model
        self.symbol_to_idx = symbol_to_idx
        self.limit_coverage = limit_coverage
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        cluster = item['cluster']
        
        if self.limit_coverage is not None:
            actual_limit = min(self.limit_coverage, len(cluster))
            cluster = cluster[:actual_limit]
            
        x_data = preprocess_cluster_to_matrix(cluster, self.seq_length)
        label_seq = item['label']
        y_data = np.array([self.symbol_to_idx[s] for s in label_seq], dtype=np.longlong)
        
        return torch.tensor(x_data, dtype=torch.float32), torch.tensor(y_data, dtype=torch.long)


In [7]:
# =============================================================================
# CELL 8: NEURAL NETWORK MODEL (Bi-LSTM)
# =============================================================================

class CompositeDecoderLSTM(nn.Module):
    """
    Bidirectional LSTM Decoder for Composite DNA.
    
    Input: (Batch, 4, L) frequency matrix
    Output: (Batch, vocab_size, L) logits
    """
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        # x: (Batch, 4, L) -> (Batch, L, 4)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        # Return: (Batch, vocab_size, L)
        return logits.permute(0, 2, 1)

In [8]:
# =============================================================================
# CELL 9: BASELINE DECODERS
# =============================================================================

def min_distance_decoder(obs, ideal_vectors):
    """
    Minimum Euclidean Distance Decoder (L2 norm).
    
    Args:
        obs: (Batch, L, 4) observed frequency vectors
        ideal_vectors: (num_classes, 4) ideal frequency vectors
    
    Returns:
        (Batch, L) predicted class indices
    """
    # obs: (B, L, 4) -> (B, L, 1, 4)
    # ideal: (C, 4) -> (1, 1, C, 4)
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """
    KL Divergence Decoder.
    
    Minimizes cross-entropy between observed and ideal distributions.
    Uses epsilon smoothing on ideal vectors to avoid log(0).
    
    Args:
        obs: (Batch, L, 4) observed frequency vectors
        ideal_vectors: (num_classes, 4) ideal frequency vectors
        epsilon: Smoothing parameter (CRITICAL: use 0.01, not 1e-10)
    
    Returns:
        (Batch, L) predicted class indices
    """
    # Smooth ideal vectors
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    # Cross-entropy: -Σ P_obs * log(P_ideal)
    obs_expanded = obs.unsqueeze(2)  # (Batch, L, 1, 4)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)  # (1, 1, C, 4)
    
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)  # (Batch, L, C)
    
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """
    Maximum Likelihood Decoder.
    
    Maximizes log-likelihood of observed frequencies under ideal distributions.
    Uses epsilon smoothing on ideal vectors.
    
    Args:
        obs: (Batch, L, 4) observed frequency vectors
        ideal_vectors: (num_classes, 4) ideal frequency vectors
        epsilon: Smoothing parameter
    
    Returns:
        (Batch, L) predicted class indices
    """
    # Smooth ideal vectors
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    # Log-likelihood: Σ P_obs * log(P_ideal)
    obs_expanded = obs.unsqueeze(2)  # (Batch, L, 1, 4)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)  # (1, 1, C, 4)
    
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)  # (Batch, L, C)
    
    return torch.argmax(log_likelihood, dim=-1)

In [9]:
# =============================================================================
# CELL 10: EARLY STOPPING CLASS
# =============================================================================

class EarlyStopping:
    """Early stopping with patience and best model saving."""
    
    def __init__(self, patience=5, path='checkpoint.pt', verbose=True):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.path = path
        self.verbose = verbose
        self.best_val_loss = float('inf')

    def __call__(self, val_loss, model):
        score = -val_loss
        
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f"      EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
            
    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f"      ✓ Val loss improved ({self.best_val_loss:.4f} → {val_loss:.4f}). Saving...")
        torch.save(model.state_dict(), self.path)
        self.best_val_loss = val_loss

In [10]:
# =============================================================================
# CELL 11: TRAINING FUNCTION
# =============================================================================

def train_model(model, train_loader, val_loader, config, weights_path, device):
    """
    Train the model with warmup + cosine annealing scheduler.
    
    Args:
        model: Neural network model
        train_loader: Training data loader
        val_loader: Validation data loader
        config: Configuration dictionary
        weights_path: Path to save best model weights
        device: torch device
    
    Returns:
        Dictionary with training history (train_loss, val_loss, lr)
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Schedulers: Linear warmup + Cosine annealing
    warmup_scheduler = LinearLR(
        optimizer, 
        start_factor=0.1, 
        total_iters=config['warmup_epochs']
    )
    cosine_scheduler = CosineAnnealingLR(
        optimizer, 
        T_max=config['epochs'] - config['warmup_epochs'],
        eta_min=config['min_lr']
    )
    scheduler = SequentialLR(
        optimizer, 
        schedulers=[warmup_scheduler, cosine_scheduler], 
        milestones=[config['warmup_epochs']]
    )
    
    early_stopper = EarlyStopping(
        patience=config['patience'], 
        path=weights_path,
        verbose=True
    )
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'lr': []
    }
    
    print(f"\n   🏋️ Training Configuration:")
    print(f"      Epochs: {config['epochs']}, Patience: {config['patience']}")
    print(f"      Warmup: {config['warmup_epochs']} epochs")
    print(f"      LR: {config['learning_rate']} → {config['min_lr']}")
    
    for epoch in range(config['epochs']):
        start_time = time.time()
        
        # --- TRAINING PHASE ---
        model.train()
        train_loss_accum = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss_accum += loss.item()
            
        avg_train_loss = train_loss_accum / len(train_loader)
        
        # --- VALIDATION PHASE ---
        model.eval()
        val_loss_accum = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss_accum += criterion(outputs, labels).item()
        
        avg_val_loss = val_loss_accum / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start_time
        
        # Logging
        print(f"   Epoch {epoch+1:03d}/{config['epochs']} | "
              f"Train: {avg_train_loss:.4f} | "
              f"Val: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.2e} | "
              f"Time: {elapsed:.1f}s")
        
        # Step scheduler & Early stopping
        scheduler.step()
        early_stopper(avg_val_loss, model)
        
        if early_stopper.early_stop:
            print(f"\n   🛑 Early stopping triggered at epoch {epoch+1}")
            break
    
    return history

In [11]:
# =============================================================================
# CELL 12: EVALUATION FUNCTION
# =============================================================================

def evaluate_all_decoders(model, loader, ideal_vectors, device):
    """
    Evaluate all 4 decoders on the given data loader.
    
    Args:
        model: Trained Bi-LSTM model
        loader: Data loader
        ideal_vectors: Ideal frequency vectors for baseline decoders
        device: torch device
    
    Returns:
        Dictionary with accuracy for each decoder
    """
    model.eval()
    
    correct = {'lstm': 0, 'mindist': 0, 'kl': 0, 'ml': 0}
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            obs = inputs.permute(0, 2, 1)  # (Batch, L, 4)
            
            # 1. LSTM Decoder
            outputs = model(inputs)
            pred_lstm = torch.argmax(outputs, dim=1)
            
            # 2. Minimum Distance Decoder
            pred_mindist = min_distance_decoder(obs, ideal_vectors)
            
            # 3. KL Divergence Decoder
            pred_kl = kl_divergence_decoder(obs, ideal_vectors)
            
            # 4. Maximum Likelihood Decoder
            pred_ml = maximum_likelihood_decoder(obs, ideal_vectors)
            
            # Count correct
            total += labels.numel()
            correct['lstm'] += (pred_lstm == labels).sum().item()
            correct['mindist'] += (pred_mindist == labels).sum().item()
            correct['kl'] += (pred_kl == labels).sum().item()
            correct['ml'] += (pred_ml == labels).sum().item()
    
    accuracies = {k: 100 * v / total for k, v in correct.items()}
    return accuracies

In [12]:
# =============================================================================
# CELL 13: FULL EXPERIMENT FOR SINGLE COVERAGE
# =============================================================================

def run_experiment_for_coverage(coverage_M, config, symbol_to_idx, ideal_vectors, device):
    """
    Run complete experiment for a single coverage level.
    Paths are dynamically generated based on error model and alphabet.
    """
    print(f"\n{'='*70}")
    print(f"🔬 EXPERIMENT FOR COVERAGE M = {coverage_M}")
    print(f"   Error Model: {config['error_name']}, Seq Length: {config['seq_length']}")
    print(f"{'='*70}")
    
    # 1. Prepare Data
    set_seed(config['seed'])
    
    full_ds = CompositeDNADataset(
        config['dataset_path'], 
        config['seq_length'],  # Variable sequence length
        symbol_to_idx,
        limit_coverage=coverage_M
    )
    
    train_size = int(0.8 * len(full_ds))
    val_size = len(full_ds) - train_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config['batch_size'], shuffle=False, num_workers=0)
    
    print(f"   📊 Data: {train_size:,} train | {val_size:,} validation")
    print(f"   📏 Sequence length: {config['seq_length']}")
    
    # 2. Setup Model
    model = CompositeDecoderLSTM(config).to(device)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🧠 Model: {config['vocab_size']} classes, {num_params:,} parameters")
    
    # Paths with error model and alphabet in filename
    model_prefix = f"{config['error_name']}_{config['alphabet_mode']}"
    best_weights_path = os.path.join(config['results_dir'], f"best_model_{model_prefix}_M{coverage_M}.pth")
    final_weights_path = os.path.join(config['results_dir'], f"final_model_{model_prefix}_M{coverage_M}.pth")
    history_path = os.path.join(config['results_dir'], f"training_history_{model_prefix}_M{coverage_M}.json")
    
    # 3. Train
    history = train_model(model, train_loader, val_loader, config, best_weights_path, device)
    
    # Save final model and history
    torch.save(model.state_dict(), final_weights_path)
    print(f"   💾 Final model saved: {final_weights_path}")
    
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=4)
    print(f"   📊 Training history saved: {history_path}")
    
    # 4. Load Best Model & Evaluate
    print(f"\n   📈 Evaluating all decoders...")
    model.load_state_dict(torch.load(best_weights_path, map_location=device))
    
    accuracies = evaluate_all_decoders(model, val_loader, ideal_vectors, device)
    
    print(f"\n   ✅ RESULTS M={coverage_M} ({config['error_name']}):")
    print(f"      Bi-LSTM:         {accuracies['lstm']:.2f}%")
    print(f"      Min. Distance:   {accuracies['mindist']:.2f}%")
    print(f"      KL Divergence:   {accuracies['kl']:.2f}%")
    print(f"      Max. Likelihood: {accuracies['ml']:.2f}%")
    
    return accuracies, history

In [13]:
# =============================================================================
# CELL 14: PLOTTING FUNCTIONS
# =============================================================================

def plot_training_history(history, coverage_M, save_path, config):
    """Plot training and validation loss curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss plot
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'Training & Validation Loss (M={coverage_M}, {config["error_name"]})', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Learning rate plot
    ax2.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Learning Rate', fontsize=12)
    ax2.set_title(f'Learning Rate Schedule (M={coverage_M})', fontsize=14)
    ax2.set_yscale('log')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   📈 Training plot saved: {save_path}")


def plot_comparison_results(results, save_path, config):
    """Plot comparison of all decoders across coverage levels."""
    plt.figure(figsize=(12, 7))
    
    plt.plot(results['coverage'], results['lstm'], 
             'o-', lw=2.5, ms=8, c='#2ecc71', label='Bi-LSTM (Ours)')
    plt.plot(results['coverage'], results['mindist'], 
             's--', lw=2.5, ms=8, c='#e74c3c', label='Min. Distance')
    plt.plot(results['coverage'], results['kl'], 
             '^-.', lw=2.5, ms=8, c='#3498db', label='KL Divergence')
    plt.plot(results['coverage'], results['ml'], 
             'd:', lw=2.5, ms=8, c='#9b59b6', label='Max. Likelihood')
    
    title = (f"Composite DNA Decoding: {config['error_name']} Error Model\n"
             f"({config['alphabet_mode']}: {config['vocab_size']} classes, "
             f"Seq Length: {config['seq_length']})")
    
    plt.xlabel("Coverage Depth (M)", fontsize=12)
    plt.ylabel("Symbol Accuracy (%)", fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)
    plt.xticks(results['coverage'])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📈 Comparison plot saved: {save_path}")

In [14]:
# =============================================================================
# CELL 15: VERIFY DATASET EXISTS
# =============================================================================

print("\n" + "="*70)
print("📦 LOADING DATASET")
print("="*70)

if not os.path.exists(CONFIG['dataset_path']):
    raise FileNotFoundError(
        f"\n❌ Dataset not found: {CONFIG['dataset_path']}\n"
        f"   Please run dataset generator with:\n"
        f"   ERROR_MODEL = '{CONFIG['error_model']}'\n"
        f"   ALPHABET_MODE = '{CONFIG['alphabet_mode']}'"
    )

# Load and display metadata
with open(CONFIG['dataset_path'], 'rb') as f:
    data = pickle.load(f)

print(f"✅ Dataset loaded: {CONFIG['dataset_path']}")
print(f"   Samples: {len(data['data']):,}")
print(f"   Sequence Length: {data['metadata']['seq_length']}")
print(f"   Error Model: {CONFIG['error_name']}")
print(f"\n   Metadata:")
for key, value in data['metadata'].items():
    if key not in ['symbol_to_idx', 'symbols', 'ideal_vectors']:
        print(f"      {key}: {value}")

# In[16]:

# =============================================================================
# CELL 16: BUILD SYMBOL MAPPINGS
# =============================================================================

SYMBOL_TO_IDX = build_symbol_to_idx(CONFIG["alphabet_mode"])
IDX_TO_SYMBOL = {v: k for k, v in SYMBOL_TO_IDX.items()}
IDEAL_VECTORS = build_ideal_vectors(CONFIG["alphabet_mode"]).to(device)

print(f"\n📊 Symbol Mappings ({CONFIG['alphabet_mode']}):")
print(f"   Total symbols: {len(SYMBOL_TO_IDX)}")


📦 LOADING DATASET
✅ Dataset loaded: ./dataset/dna_G15_2mix_only_100000_25.pkl
   Samples: 100,000
   Sequence Length: 104
   Error Model: G15

   Metadata:
      type: Composite DNA (2mix_only)
      error_profile: Erlich (EZ17)
      num_samples: 100000
      seq_length: 104
      coverage_depth: 25
      vocab_size: 10
      alphabet_mode: 2mix_only
      timestamp: 2025-12-15 04:30:41
      seed: 42

📊 Symbol Mappings (2mix_only):
   Total symbols: 10


In [15]:
# =============================================================================
# CELL 17: MAIN EXECUTION - RUN ALL EXPERIMENTS
# =============================================================================

print("\n" + "="*70)
print("🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS")
print("="*70)
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Sequence Length: {CONFIG['seq_length']}")
print(f"   Coverage Levels: {CONFIG['coverage_levels']}")
print(f"   Alphabet: {CONFIG['alphabet_mode']} ({CONFIG['vocab_size']} classes)")

results = {
    'coverage': CONFIG['coverage_levels'],
    'lstm': [],
    'mindist': [],
    'kl': [],
    'ml': [],
    'config': {
        'error_model': CONFIG['error_model'],
        'error_name': CONFIG['error_name'],
        'seq_length': CONFIG['seq_length'],
        'alphabet_mode': CONFIG['alphabet_mode'],
        'vocab_size': CONFIG['vocab_size'],
        'hidden_dim': CONFIG['hidden_dim'],
        'num_layers': CONFIG['num_layers'],
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
}

all_histories = {}

for M in CONFIG['coverage_levels']:
    accuracies, history = run_experiment_for_coverage(
        M, CONFIG, SYMBOL_TO_IDX, IDEAL_VECTORS, device
    )
    
    results['lstm'].append(accuracies['lstm'])
    results['mindist'].append(accuracies['mindist'])
    results['kl'].append(accuracies['kl'])
    results['ml'].append(accuracies['ml'])
    all_histories[M] = history
    
    # Plot training history
    plot_prefix = f"{CONFIG['error_name']}_{CONFIG['alphabet_mode']}"
    plot_path = os.path.join(CONFIG['results_dir'], f"training_plot_{plot_prefix}_M{M}.png")
    plot_training_history(history, M, plot_path, CONFIG)


🚀 RUNNING EXPERIMENTS FOR ALL COVERAGE LEVELS
   Error Model: grass (G15)
   Sequence Length: 104
   Coverage Levels: [1, 2, 3, 5, 8, 10, 15, 20, 25]
   Alphabet: 2mix_only (10 classes)

🔬 EXPERIMENT FOR COVERAGE M = 1
   Error Model: G15, Seq Length: 104
   📊 Data: 80,000 train | 20,000 validation
   📏 Sequence length: 104
   🧠 Model: 10 classes, 535,050 parameters

   🏋️ Training Configuration:
      Epochs: 100, Patience: 10
      Warmup: 10 epochs
      LR: 0.001 → 1e-06
   Epoch 001/100 | Train: 2.2341 | Val: 2.0573 | LR: 1.00e-04 | Time: 34.2s
      ✓ Val loss improved (inf → 2.0573). Saving...
   Epoch 002/100 | Train: 1.6424 | Val: 1.4661 | LR: 1.90e-04 | Time: 33.7s
      ✓ Val loss improved (2.0573 → 1.4661). Saving...
   Epoch 003/100 | Train: 1.4722 | Val: 1.4598 | LR: 2.80e-04 | Time: 33.7s
      ✓ Val loss improved (1.4661 → 1.4598). Saving...
   Epoch 004/100 | Train: 1.4637 | Val: 1.4558 | LR: 3.70e-04 | Time: 34.6s
      ✓ Val loss improved (1.4598 → 1.4558). Saving..

/homes/shubham/anaconda3/envs/pytorchenv/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:149: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


   Epoch 011/100 | Train: 1.4482 | Val: 1.4455 | LR: 1.00e-03 | Time: 32.9s
      ✓ Val loss improved (1.4459 → 1.4455). Saving...
   Epoch 012/100 | Train: 1.4477 | Val: 1.4453 | LR: 1.00e-03 | Time: 33.5s
      ✓ Val loss improved (1.4455 → 1.4453). Saving...
   Epoch 013/100 | Train: 1.4471 | Val: 1.4443 | LR: 9.99e-04 | Time: 33.2s
      ✓ Val loss improved (1.4453 → 1.4443). Saving...
   Epoch 014/100 | Train: 1.4465 | Val: 1.4444 | LR: 9.97e-04 | Time: 34.6s
      EarlyStopping counter: 1/10
   Epoch 015/100 | Train: 1.4465 | Val: 1.4441 | LR: 9.95e-04 | Time: 34.1s
      ✓ Val loss improved (1.4443 → 1.4441). Saving...
   Epoch 016/100 | Train: 1.4462 | Val: 1.4439 | LR: 9.92e-04 | Time: 33.3s
      ✓ Val loss improved (1.4441 → 1.4439). Saving...
   Epoch 017/100 | Train: 1.4460 | Val: 1.4441 | LR: 9.89e-04 | Time: 33.2s
      EarlyStopping counter: 1/10
   Epoch 018/100 | Train: 1.4457 | Val: 1.4437 | LR: 9.85e-04 | Time: 33.7s
      ✓ Val loss improved (1.4439 → 1.4437). Savi

   Epoch 080/100 | Train: 1.4320 | Val: 1.4350 | LR: 1.29e-04 | Time: 32.0s
      EarlyStopping counter: 5/10
   Epoch 081/100 | Train: 1.4318 | Val: 1.4352 | LR: 1.18e-04 | Time: 32.3s
      EarlyStopping counter: 6/10
   Epoch 082/100 | Train: 1.4316 | Val: 1.4356 | LR: 1.07e-04 | Time: 31.7s
      EarlyStopping counter: 7/10
   Epoch 083/100 | Train: 1.4315 | Val: 1.4352 | LR: 9.64e-05 | Time: 32.2s
      EarlyStopping counter: 8/10
   Epoch 084/100 | Train: 1.4314 | Val: 1.4353 | LR: 8.64e-05 | Time: 33.6s
      EarlyStopping counter: 9/10
   Epoch 085/100 | Train: 1.4311 | Val: 1.4356 | LR: 7.69e-05 | Time: 33.4s
      EarlyStopping counter: 10/10

   🛑 Early stopping triggered at epoch 85
   💾 Final model saved: ./results_G15_2mix_only/final_model_G15_2mix_only_M1.pth
   📊 Training history saved: ./results_G15_2mix_only/training_history_G15_2mix_only_M1.json

   📈 Evaluating all decoders...

   ✅ RESULTS M=1 (G15):
      Bi-LSTM:         38.81%
      Min. Distance:   38.81%
     

   Epoch 054/100 | Train: 0.8895 | Val: 0.8925 | LR: 5.35e-04 | Time: 53.0s
      EarlyStopping counter: 1/10
   Epoch 055/100 | Train: 0.8889 | Val: 0.8926 | LR: 5.18e-04 | Time: 53.2s
      EarlyStopping counter: 2/10
   Epoch 056/100 | Train: 0.8887 | Val: 0.8924 | LR: 5.00e-04 | Time: 52.7s
      EarlyStopping counter: 3/10
   Epoch 057/100 | Train: 0.8888 | Val: 0.8925 | LR: 4.83e-04 | Time: 52.8s
      EarlyStopping counter: 4/10
   Epoch 058/100 | Train: 0.8882 | Val: 0.8921 | LR: 4.66e-04 | Time: 53.4s
      ✓ Val loss improved (0.8923 → 0.8921). Saving...
   Epoch 059/100 | Train: 0.8878 | Val: 0.8934 | LR: 4.48e-04 | Time: 53.2s
      EarlyStopping counter: 1/10
   Epoch 060/100 | Train: 0.8877 | Val: 0.8919 | LR: 4.31e-04 | Time: 53.5s
      ✓ Val loss improved (0.8921 → 0.8919). Saving...
   Epoch 061/100 | Train: 0.8874 | Val: 0.8919 | LR: 4.14e-04 | Time: 53.1s
      EarlyStopping counter: 1/10
   Epoch 062/100 | Train: 0.8873 | Val: 0.8916 | LR: 3.97e-04 | Time: 53.5s
  

...

   Epoch 060/100 | Train: 0.0312 | Val: 0.0329 | LR: 4.31e-04 | Time: 225.2s
      EarlyStopping counter: 1/10
   Epoch 061/100 | Train: 0.0311 | Val: 0.0328 | LR: 4.14e-04 | Time: 228.0s
      ✓ Val loss improved (0.0329 → 0.0328). Saving...
   Epoch 062/100 | Train: 0.0310 | Val: 0.0327 | LR: 3.97e-04 | Time: 224.8s
      ✓ Val loss improved (0.0328 → 0.0327). Saving...
   Epoch 063/100 | Train: 0.0308 | Val: 0.0328 | LR: 3.80e-04 | Time: 230.6s
      EarlyStopping counter: 1/10
   Epoch 064/100 | Train: 0.0307 | Val: 0.0328 | LR: 3.63e-04 | Time: 225.5s
      EarlyStopping counter: 2/10
   Epoch 065/100 | Train: 0.0306 | Val: 0.0327 | LR: 3.46e-04 | Time: 225.2s
      ✓ Val loss improved (0.0327 → 0.0327). Saving...
   Epoch 066/100 | Train: 0.0305 | Val: 0.0327 | LR: 3.30e-04 | Time: 226.8s
      EarlyStopping counter: 1/10
   Epoch 067/100 | Train: 0.0304 | Val: 0.0327 | LR: 3.13e-04 | Time: 229.0s
      EarlyStopping counter: 2/10
   Epoch 068/100 | Train: 0.0303 | Val: 0.0326 | 

   Epoch 020/100 | Train: 0.0115 | Val: 0.0094 | LR: 9.76e-04 | Time: 344.4s
      EarlyStopping counter: 1/10
   Epoch 021/100 | Train: 0.0110 | Val: 0.0089 | LR: 9.70e-04 | Time: 366.0s
      ✓ Val loss improved (0.0093 → 0.0089). Saving...
   Epoch 022/100 | Train: 0.0107 | Val: 0.0086 | LR: 9.64e-04 | Time: 331.4s
      ✓ Val loss improved (0.0089 → 0.0086). Saving...
   Epoch 023/100 | Train: 0.0103 | Val: 0.0084 | LR: 9.57e-04 | Time: 330.6s
      ✓ Val loss improved (0.0086 → 0.0084). Saving...
   Epoch 024/100 | Train: 0.0100 | Val: 0.0084 | LR: 9.49e-04 | Time: 331.1s
      EarlyStopping counter: 1/10
   Epoch 025/100 | Train: 0.0097 | Val: 0.0083 | LR: 9.42e-04 | Time: 329.5s
      ✓ Val loss improved (0.0084 → 0.0083). Saving...
   Epoch 026/100 | Train: 0.0094 | Val: 0.0079 | LR: 9.33e-04 | Time: 343.3s
      ✓ Val loss improved (0.0083 → 0.0079). Saving...
   Epoch 027/100 | Train: 0.0093 | Val: 0.0082 | LR: 9.24e-04 | Time: 331.7s
      EarlyStopping counter: 1/10
   Epoc

   Epoch 004/100 | Train: 0.0471 | Val: 0.0245 | LR: 3.70e-04 | Time: 431.5s
      ✓ Val loss improved (0.0557 → 0.0245). Saving...
   Epoch 005/100 | Train: 0.0252 | Val: 0.0148 | LR: 4.60e-04 | Time: 478.5s
      ✓ Val loss improved (0.0245 → 0.0148). Saving...
   Epoch 006/100 | Train: 0.0169 | Val: 0.0106 | LR: 5.50e-04 | Time: 436.3s
      ✓ Val loss improved (0.0148 → 0.0106). Saving...
   Epoch 007/100 | Train: 0.0128 | Val: 0.0082 | LR: 6.40e-04 | Time: 440.3s
      ✓ Val loss improved (0.0106 → 0.0082). Saving...
   Epoch 008/100 | Train: 0.0104 | Val: 0.0070 | LR: 7.30e-04 | Time: 432.0s
      ✓ Val loss improved (0.0082 → 0.0070). Saving...
   Epoch 009/100 | Train: 0.0087 | Val: 0.0060 | LR: 8.20e-04 | Time: 432.9s
      ✓ Val loss improved (0.0070 → 0.0060). Saving...
   Epoch 010/100 | Train: 0.0076 | Val: 0.0051 | LR: 9.10e-04 | Time: 434.2s
      ✓ Val loss improved (0.0060 → 0.0051). Saving...
   Epoch 011/100 | Train: 0.0066 | Val: 0.0045 | LR: 1.00e-03 | Time: 469.2s

   Epoch 072/100 | Train: 0.0009 | Val: 0.0015 | LR: 2.36e-04 | Time: 428.9s
      EarlyStopping counter: 8/10
   Epoch 073/100 | Train: 0.0009 | Val: 0.0015 | LR: 2.21e-04 | Time: 431.9s
      EarlyStopping counter: 9/10
   Epoch 074/100 | Train: 0.0009 | Val: 0.0014 | LR: 2.07e-04 | Time: 432.8s
      EarlyStopping counter: 10/10

   🛑 Early stopping triggered at epoch 74
   💾 Final model saved: ./results_G15_2mix_only/final_model_G15_2mix_only_M20.pth
   📊 Training history saved: ./results_G15_2mix_only/training_history_G15_2mix_only_M20.json

   📈 Evaluating all decoders...

   ✅ RESULTS M=20 (G15):
      Bi-LSTM:         99.96%
      Min. Distance:   97.90%
      KL Divergence:   99.27%
      Max. Likelihood: 99.27%
   📈 Training plot saved: ./results_G15_2mix_only/training_plot_G15_2mix_only_M20.png

🔬 EXPERIMENT FOR COVERAGE M = 25
   Error Model: G15, Seq Length: 104
   📊 Data: 80,000 train | 20,000 validation
   📏 Sequence length: 104
   🧠 Model: 10 classes, 535,050 parameters

   Epoch 058/100 | Train: 0.0004 | Val: 0.0005 | LR: 4.66e-04 | Time: 531.6s
      EarlyStopping counter: 1/10
   Epoch 059/100 | Train: 0.0004 | Val: 0.0005 | LR: 4.48e-04 | Time: 529.8s
      EarlyStopping counter: 2/10
   Epoch 060/100 | Train: 0.0004 | Val: 0.0005 | LR: 4.31e-04 | Time: 531.3s
      ✓ Val loss improved (0.0005 → 0.0005). Saving...
   Epoch 061/100 | Train: 0.0004 | Val: 0.0005 | LR: 4.14e-04 | Time: 549.8s
      EarlyStopping counter: 1/10
   Epoch 062/100 | Train: 0.0004 | Val: 0.0005 | LR: 3.97e-04 | Time: 530.0s
      EarlyStopping counter: 2/10
   Epoch 063/100 | Train: 0.0004 | Val: 0.0005 | LR: 3.80e-04 | Time: 534.4s
      EarlyStopping counter: 3/10
   Epoch 064/100 | Train: 0.0003 | Val: 0.0005 | LR: 3.63e-04 | Time: 537.1s
      ✓ Val loss improved (0.0005 → 0.0005). Saving...
   Epoch 065/100 | Train: 0.0003 | Val: 0.0005 | LR: 3.46e-04 | Time: 553.5s
      EarlyStopping counter: 1/10
   Epoch 066/100 | Train: 0.0003 | Val: 0.0005 | LR: 3.30e-04 | Time: 

In [16]:
# =============================================================================
# CELL 18: SAVE FINAL RESULTS & PLOT
# =============================================================================

print("\n" + "="*70)
print("📊 FINAL RESULTS SUMMARY")
print("="*70)

# Save results to JSON
results_json_path = os.path.join(CONFIG['results_dir'], "experiment_results.json")
with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=4)
print(f"💾 Results saved: {results_json_path}")

# Print table
print(f"\n   Error Model: {CONFIG['error_name']}, Seq Length: {CONFIG['seq_length']}")
print(f"   {'M':<8} {'Bi-LSTM':<12} {'Min.Dist':<12} {'KL Div':<12} {'Max.Like':<12}")
print(f"   {'-'*56}")
for i, M in enumerate(results['coverage']):
    print(f"   {M:<8} {results['lstm'][i]:<12.2f} {results['mindist'][i]:<12.2f} "
          f"{results['kl'][i]:<12.2f} {results['ml'][i]:<12.2f}")
print(f"   {'='*56}")

# Final comparison plot
plot_path = os.path.join(CONFIG['results_dir'], "final_comparison_plot.png")
plot_comparison_results(results, plot_path, CONFIG)

print(f"\n✅ All experiments completed!")
print(f"   Error Model: {CONFIG['error_model']} ({CONFIG['error_name']})")
print(f"   Results directory: {CONFIG['results_dir']}")


📊 FINAL RESULTS SUMMARY
💾 Results saved: ./results_G15_2mix_only/experiment_results.json

   Error Model: G15, Seq Length: 104
   M        Bi-LSTM      Min.Dist     KL Div       Max.Like    
   --------------------------------------------------------
   1        38.81        38.81        38.81        38.81       
   2        67.39        66.43        66.43        66.43       
   3        82.40        79.45        79.45        79.45       
   5        94.30        76.73        88.62        88.62       
   8        98.37        83.63        94.21        94.21       
   10       98.95        93.30        97.13        97.13       
   15       99.84        97.56        99.09        99.09       
   20       99.96        97.90        99.27        99.27       
   25       99.99        99.25        99.74        99.74       
📈 Comparison plot saved: ./results_G15_2mix_only/final_comparison_plot.png

✅ All experiments completed!
   Error Model: grass (G15)
   Results directory: ./results_G15_2mi

In [17]:
# =============================================================================
# CELL 19: CROSS-ERROR MODEL COMPARISON (Optional)
# =============================================================================

def compare_error_models(base_dir="./", alphabet_mode="2mix_3mix_4mix"):
    """
    Compare results across different error models for the same alphabet.
    Run after training on all three error models.
    """
    error_models = ["erlich", "grass", "organick"]
    all_results = {}
    
    for error_model in error_models:
        error_name = ERROR_MODEL_SPECS[error_model]["name"]
        results_dir = f"{base_dir}/results_{error_name}_{alphabet_mode}"
        results_path = os.path.join(results_dir, "experiment_results.json")
        
        if os.path.exists(results_path):
            with open(results_path, 'r') as f:
                all_results[error_model] = json.load(f)
            print(f"✅ Loaded results for {error_model}")
        else:
            print(f"⚠️ Results not found for {error_model}")
    
    if len(all_results) < 2:
        print("Need at least 2 error models to compare")
        return
    
    # Create comparison plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    colors = {'erlich': '#2ecc71', 'grass': '#3498db', 'organick': '#e74c3c'}
    
    # Plot 1: LSTM comparison across error models
    for error_model, results in all_results.items():
        ax1.plot(results['coverage'], results['lstm'], 
                'o-', lw=2.5, ms=8, c=colors[error_model], 
                label=f"{error_model.upper()} (L={results['config']['seq_length']})")
    
    ax1.set_xlabel("Coverage Depth (M)", fontsize=12)
    ax1.set_ylabel("Symbol Accuracy (%)", fontsize=12)
    ax1.set_title(f"Bi-LSTM Performance: Error Model Comparison\n({alphabet_mode})", fontsize=14)
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 105)
    
    # Plot 2: Best decoder comparison
    ax2.bar(['Erlich', 'Grass', 'Organick'], 
            [max(all_results.get(m, {}).get('lstm', [0])) for m in error_models],
            color=[colors[m] for m in error_models])
    ax2.set_ylabel("Max Symbol Accuracy (%)", fontsize=12)
    ax2.set_title(f"Peak Performance by Error Model", fontsize=14)
    ax2.set_ylim(0, 105)
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    save_path = f"{base_dir}/comparison_error_models_{alphabet_mode}.png"
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"📊 Error model comparison saved: {save_path}")

# Uncomment to run after training all models:
# compare_error_models(alphabet_mode=CONFIG['alphabet_mode'])